In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import joblib

In [ ]:
CLEAN_TMDB_FILE_PATH = "../datasets/clean/tmdb-movies/TMDB_movie_dataset_v11.csv"
CLEAN_MOVIELENS_RATINGS_PATH = "../datasets/clean/ml-32m/ratings.csv"

ML_API_TF_IDF_MATRIX_PATH = "../../ml-api/model/tf_idf_matrix.pkl"

In [ ]:
tmdb = pd.read_csv(CLEAN_TMDB_FILE_PATH)
ratings = pd.read_csv(CLEAN_MOVIELENS_RATINGS_PATH)

In [ ]:
tmdb_id_to_index = pd.Series(tmdb.index, index=tmdb["id"]).to_dict()
ratings["rating"] = ratings["rating"] - ratings["rating"].mean()

In [ ]:
plt.hist(ratings["rating"], bins=50)
plt.title("Distribution ratings")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
print("min:", ratings["rating"].min())
print("max:", ratings["rating"].max())
print("mean:", ratings["rating"].mean())
print("std:", ratings["rating"].std())
print(np.percentile(ratings["rating"], [0, 25, 50, 75, 100]))

In [ ]:
def find_in_dataset_by_substring(movie_names):
    found = []
    for movie_name in movie_names:
        results = tmdb[tmdb["title"].str.contains(movie_name, case=False, na=False)]
        haa = results.sort_values(by="popularity", ascending=False)[["id", "title"]]
        found.append(haa.values.tolist())
    return found

def find_in_dataset_by_id(movie_id):
    return tmdb[tmdb["id"] == movie_id]["title"].values[0]

def print_ratings_dict(ratings_dict: dict[int, int]):
    for id, rating in ratings_dict.items():
        print(f"id: {id}, name: {find_in_dataset_by_id(id)}, rating: {rating}")

In [ ]:
find_in_dataset_by_substring(["justice", "batman", "superman", "flash", "lantern", "steel", "watchmen", "joker"])

In [ ]:
find_in_dataset_by_id(791373)

In [ ]:
marvel_fan_rd = {
  569094: 4.5,
  634649: 4,
  271110: 3,
  1771: 4,
  10138: 4.5,
  1724: 3.5,
  26881: 4,
  299536: 5,
  9320: 4.5
}

print_ratings_dict(marvel_fan_rd)
print()
print()
print()

marvel_fan_dc_hater_rd = {
  569094: 4.5,
  634649: 4,
  271110: 3,
  1771: 4,
  10138: 4.5,
  1724: 3.5,
  26881: 4,
  299536: 5,
  9320: 4.5,
  141052: 1,
  209112: 0,
  414906: 2.5,
  272: 1.4,
  1924: 1,
  298618: 0.7,
  44912: 0,
  49521: 1.8,
  13183: 0.3,
  475557: 3
}

print_ratings_dict(marvel_fan_dc_hater_rd)

Content based filtering

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(tmdb["content"])

In [ ]:
def build_user_profile_content(ratings_dict):
    vectors = []
    weights = []

    for tmdb_id, rating in ratings_dict.items():
        if tmdb_id in tmdb_id_to_index:
            idx = tmdb_id_to_index[tmdb_id]
            vectors.append(tfidf_matrix[idx].toarray()[0])
            weights.append(rating - 2.5)

    vectors = np.array(vectors)
    weights = np.array(weights)

    user_profile = np.sum(vectors * weights[:, np.newaxis], axis=0)
    user_profile = user_profile.reshape(1, -1)

    return user_profile

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_content(user_vector, top_k=20):
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    weighted_sims = sims * tmdb["popularity_log"].values
    top_idx = weighted_sims.argsort()[-top_k:][::-1]
    return tmdb.iloc[top_idx][["id", "title", "vote_average", "popularity"]]

def find_recommended_content(user_vector, movie_id=1): # top_k tu asi nepotřebujeme, když hledáš konkrétní ID
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    weighted_sims = sims * tmdb["popularity_log"].values
    results = tmdb[["id", "title", "vote_average", "popularity"]].copy()
    results["match_score"] = weighted_sims

    # 1. Přidáme absolutní pořadí (1 = úplně nejlepší film pro tohoto uživatele)
    # ascending=False znamená, že nejvyšší skóre dostane rank 1
    results["rank"] = results["match_score"].rank(ascending=False).astype(int)
    
    # 2. Přidáme percentil (např. 99.5 znamená, že film je lepší než 99.5 % datasetu)
    # pct=True nám vrátí hodnoty od 0 do 1, takže to vynásobíme 100
    results["percentile"] = results["match_score"].rank(pct=True) * 100

    return results[results["id"] == movie_id]

def analyze_recommended_content(user_vector):
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    weighted_sims = sims * tmdb["popularity_log"].values
    
    results = tmdb[["id", "title", "vote_average", "popularity"]].copy()
    results["match_score"] = weighted_sims

    # --- 1. Základní statistiky a percentily ---
    print("=== STATISTIKY MATCH SCORE ===")
    stats = results["match_score"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])
    print(stats)
    print("\n")

    # --- 2. Analýza distribuce (Kladné vs Záporné) ---
    print("=== DISTRIBUCE SKÓRE ===")
    kladne = (results["match_score"] > 0).sum()
    zaporne = (results["match_score"] < 0).sum()
    nuly = (results["match_score"] == 0).sum()
    
    print(f"Filmy s kladným skóre (kandidáti na doporučení): {kladne}")
    print(f"Filmy se záporným skóre (aktivně penalizované):  {zaporne}")
    print(f"Filmy s nulovým skóre (žádná shoda v textu):    {nuly}")

In [ ]:
marvel_fan = build_user_profile_content(marvel_fan_rd)

#print(recommend_content(marvel_fan))
print(find_recommended_content(marvel_fan, movie_id = 791373))
print(find_recommended_content(marvel_fan, movie_id = 569094))
analyze_recommended_content(marvel_fan)

print()
print()
print()
print()

marvel_fan_dc_hater = build_user_profile_content(marvel_fan_dc_hater_rd)

#print(recommend_content(marvel_fan_dc_hater))
print(find_recommended_content(marvel_fan_dc_hater, movie_id = 791373))
print(find_recommended_content(marvel_fan_dc_hater, movie_id = 569094))
analyze_recommended_content(marvel_fan_dc_hater)

Saving this abhorrent "progress"

In [ ]:
joblib.dump(tfidf_matrix, ML_API_TF_IDF_MATRIX_PATH)